## CELL INIT — запускати після кожного перезапуску сесії

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['apt-get','install','-y','fonts-liberation'], capture_output=True)
subprocess.run(['pip','install','xgboost','scikit-learn','joblib',
                'imbalanced-learn','-q'], capture_output=True)

import os, json, warnings, glob
import numpy as np, pandas as pd
import matplotlib, matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

BASE       = '/content/drive/MyDrive/adversarial_iot_paper'
RAW_DIR    = os.path.join(BASE, 'data')
SEC6_DIR   = os.path.join(BASE, 'results', 'section6_v2')
FIG_DIR    = os.path.join(BASE, 'figures',  'section6_v2')
MODELS_DIR = os.path.join(BASE, 'models_v2')
for d in [SEC6_DIR, FIG_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

found = [f.name for f in fm.fontManager.ttflist if 'Liberation Sans' in f.name]
FONT  = 'Liberation Sans' if found else 'DejaVu Sans'
matplotlib.rcParams.update({
    'font.family': FONT, 'font.size': 8, 'axes.linewidth': 0.6,
    'axes.titlesize': 8, 'axes.labelsize': 7.5,
    'xtick.labelsize': 7, 'ytick.labelsize': 7,
    'legend.fontsize': 6.5, 'pdf.fonttype': 42
})
FIG_W = 6.93; DPI = 300; SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

DS_TAGS = {
    'CIC-IDS 2017'       : 'CIC-IDS_2017',
    'UNSW-NB15'          : 'UNSW-NB15',
    'Gotham IoT 2025'    : 'Gotham_IoT_2025',
    'CIC-YNU-IoTMal 2026': 'CIC-YNU-IoTMal_2026',
}

# CNN архітектура (потрібна для завантаження моделей)
class CNN1D(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,128,3,padding=1), nn.BatchNorm1d(128),
            nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128,64,3,padding=1), nn.BatchNorm1d(64),
            nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64,32,3,padding=1), nn.BatchNorm1d(32),
            nn.ReLU(), nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(32,128),
            nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n_classes)
        )
    def forward(self, x): return self.head(self.conv(x.unsqueeze(1)))

def predict_batched(model, X, batch=4096):
    """Inference по батчах — не навантажує GPU пам'ять."""
    model.eval(); all_p = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = torch.FloatTensor(X[i:i+batch]).to(DEVICE)
            all_p.append(torch.argmax(model(xb),1).cpu().numpy())
            del xb
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return np.concatenate(all_p)

print(f'✓ INIT complete | Device: {DEVICE} | Font: {FONT}')

## CELL STATUS — перевірка файлів на Drive

In [ ]:
print('=== Section 6 v2 STATUS ===')
for ds_name, tag in DS_TAGS.items():
    files = {}
    for prefix in ['X_tr','X_te','y_tr','y_te']:
        p = os.path.join(SEC6_DIR, f'{prefix}_{tag}.npy')
        files[prefix] = os.path.exists(p)
    for mn in ['rf','xgb','mlp']:
        p = os.path.join(MODELS_DIR, f'{mn}_{tag}.joblib')
        files[mn] = os.path.exists(p)
    p = os.path.join(MODELS_DIR, f'cnn_{tag}.pt')
    files['cnn'] = os.path.exists(p)
    ready = sum(files.values())
    icons = {k: '✓' if v else '—' for k,v in files.items()}
    print(f'\n{ds_name} ({ready}/{len(files)}):')
    print(f'  Data: {icons["X_tr"]} X_tr  {icons["X_te"]} X_te  '
          f'{icons["y_tr"]} y_tr  {icons["y_te"]} y_te')
    print(f'  Models: {icons["rf"]} RF  {icons["xgb"]} XGB  '
          f'{icons["mlp"]} MLP  {icons["cnn"]} CNN')

p_csv = os.path.join(SEC6_DIR,'table6_v2.csv')
p_fig = os.path.join(FIG_DIR,'fig5_v2_roc_curves.png')
print(f'\nTable 6 v2: {"✓" if os.path.exists(p_csv) else "—"}')
print(f'Figure 5 v2: {"✓" if os.path.exists(p_fig) else "—"}')

## CELL 1 — Завантаження сирих даних
Кешує X_raw/y_raw щоб не перечитувати CSV щоразу.

In [ ]:
RAW_DATA = {}

def _save_raw(tag, X, y):
    np.save(os.path.join(SEC6_DIR, f'X_raw_{tag}.npy'), X)
    np.save(os.path.join(SEC6_DIR, f'y_raw_{tag}.npy'), y)

def _load_raw(tag):
    xp = os.path.join(SEC6_DIR, f'X_raw_{tag}.npy')
    yp = os.path.join(SEC6_DIR, f'y_raw_{tag}.npy')
    if os.path.exists(xp) and os.path.exists(yp):
        return np.load(xp), np.load(yp), True
    return None, None, False

# ── CIC-IDS 2017 ───────────────────────────────────────────────
tag = 'CIC-IDS_2017'
X_raw, y_raw, cached = _load_raw(tag)
if cached:
    print(f'✓ CIC-IDS 2017: loaded from cache {X_raw.shape}')
else:
    files = sorted(glob.glob(
        os.path.join(RAW_DIR,'cicids2017','MachineLearningCVE','*.csv')))
    dfs = []
    for f in files:
        df = pd.read_csv(f, low_memory=False)
        df.columns = df.columns.str.strip()
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    label_col = next(c for c in df.columns if 'label' in c.lower())
    le = LabelEncoder()
    y_raw = le.fit_transform(df[label_col].values)
    X_df = df.drop(columns=[label_col]).apply(pd.to_numeric, errors='coerce')
    X_raw = X_df.replace([np.inf,-np.inf], np.nan).fillna(0).values.astype(np.float32)
    _save_raw(tag, X_raw, y_raw)
    print(f'CIC raw saved: {X_raw.shape}')
print(f'✓ CIC-IDS 2017: {X_raw.shape}, classes={len(np.unique(y_raw))}')
RAW_DATA['CIC-IDS 2017'] = (X_raw, y_raw)

# ── UNSW-NB15 ──────────────────────────────────────────────────
# ВАЖЛИВО: файли не мають заголовків → header=None обов'язково
# Є помилка 'Backdoor'/'Backdoors' → об'єднуємо
# Файл LIST_EVENTS виключаємо
tag = 'UNSW-NB15'
X_raw, y_raw, cached = _load_raw(tag)
if cached:
    print(f'✓ UNSW-NB15: loaded from cache {X_raw.shape}')
else:
    feat_f = os.path.join(RAW_DIR,'unsw_nb15','NUSW-NB15_features.csv')
    feat_df = pd.read_csv(feat_f, encoding='latin1')
    col_names = feat_df['Name'].str.strip().tolist()

    files = sorted([
        f for f in glob.glob(os.path.join(RAW_DIR,'unsw_nb15','UNSW-NB15_*.csv'))
        if 'LIST' not in os.path.basename(f).upper()
        and 'EVENT' not in os.path.basename(f).upper()
    ])
    print(f'  Files: {[os.path.basename(f) for f in files]}')

    dfs = []
    for f in files:
        df = pd.read_csv(f, low_memory=False, header=None)
        if len(col_names) == len(df.columns):
            df.columns = col_names
        dfs.append(df)
        print(f'  {os.path.basename(f)}: {df.shape}')

    df = pd.concat(dfs, ignore_index=True)

    # Заповнюємо NaN → 'Normal', об'єднуємо Backdoor/Backdoors
    df['attack_cat'] = (df['attack_cat']
                        .astype(str).str.strip()
                        .replace({'nan':'Normal','':'Normal','0':'Normal',
                                  'Backdoor':'Backdoors'}))
    print(f'  Classes: {sorted(df["attack_cat"].unique())}')

    le = LabelEncoder()
    y_raw = le.fit_transform(df['attack_cat'].values)
    drop = [c for c in df.columns if str(c).lower() in ['label','attack_cat','id']]
    X_df = df.drop(columns=drop, errors='ignore').apply(pd.to_numeric, errors='coerce')
    X_raw = X_df.replace([np.inf,-np.inf], np.nan).fillna(0).values.astype(np.float32)
    _save_raw(tag, X_raw, y_raw)
    print(f'UNSW raw saved: {X_raw.shape}')
print(f'✓ UNSW-NB15: {X_raw.shape}, classes={len(np.unique(y_raw))}')
RAW_DATA['UNSW-NB15'] = (X_raw, y_raw)

# ── Gotham IoT 2025 ────────────────────────────────────────────
tag = 'Gotham_IoT_2025'
X_raw, y_raw, cached = _load_raw(tag)
if cached:
    print(f'✓ Gotham IoT 2025: loaded from cache {X_raw.shape}')
else:
    files = (sorted(glob.glob(os.path.join(RAW_DIR,'gotham2025','*.parquet'))) or
             sorted(glob.glob(os.path.join(RAW_DIR,'gotham2025','*.csv'))))
    dfs = []
    for f in files:
        df = (pd.read_parquet(f) if f.endswith('.parquet')
              else pd.read_csv(f, low_memory=False))
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=True)
    label_col = next((c for c in df.columns if 'label' in c.lower()), df.columns[-1])
    le = LabelEncoder()
    y_raw = le.fit_transform(df[label_col].astype(str).values)
    X_df = df.drop(columns=[label_col]).apply(pd.to_numeric, errors='coerce')
    X_raw = X_df.replace([np.inf,-np.inf], np.nan).fillna(0).values.astype(np.float32)
    _save_raw(tag, X_raw, y_raw)
    print(f'Gotham raw saved: {X_raw.shape}')
print(f'✓ Gotham IoT 2025: {X_raw.shape}, classes={len(np.unique(y_raw))}')
RAW_DATA['Gotham IoT 2025'] = (X_raw, y_raw)

# ── CIC-YNU-IoTMal 2026 ────────────────────────────────────────
tag = 'CIC-YNU-IoTMal_2026'
X_raw, y_raw, cached = _load_raw(tag)
if cached:
    print(f'✓ CIC-YNU-IoTMal 2026: loaded from cache {X_raw.shape}')
else:
    archs = ['arm','mips','mipsel','x86']
    dfs = []
    for arch in archs:
        f = os.path.join(RAW_DIR,'cic_ynu_iotmal2026', arch, 'pcap.parquet')
        if os.path.exists(f):
            df = pd.read_parquet(f); df['arch'] = arch; dfs.append(df)
    if not dfs:
        raise FileNotFoundError(f'CIC-YNU-IoTMal parquet not found in {RAW_DIR}')
    df = pd.concat(dfs, ignore_index=True)
    label_col = next(
        (c for c in df.columns if 'label' in c.lower() or 'family' in c.lower()),
        df.columns[-1])
    le = LabelEncoder()
    y_raw = le.fit_transform(df[label_col].astype(str).values)
    drop = list({label_col,'arch'} |
                {c for c in df.columns if 'label' in c.lower() or 'family' in c.lower()})
    X_df = df.drop(columns=drop, errors='ignore').apply(pd.to_numeric, errors='coerce')
    X_raw = X_df.replace([np.inf,-np.inf], np.nan).fillna(0).values.astype(np.float32)
    _save_raw(tag, X_raw, y_raw)
    print(f'IoTMal raw saved: {X_raw.shape}')
print(f'✓ CIC-YNU-IoTMal 2026: {X_raw.shape}, classes={len(np.unique(y_raw))}')
RAW_DATA['CIC-YNU-IoTMal 2026'] = (X_raw, y_raw)

print(f'\n✓ RAW_DATA ready: {len(RAW_DATA)}/4 datasets')

## CELL 2 — Правильний pipeline (без data leakage)
Порядок: **split → variance filter → scaler.fit(train) → correlation filter → SMOTE(train only) → save**

In [ ]:
DATA = {}

for ds_name, tag in DS_TAGS.items():
    xtr_p = os.path.join(SEC6_DIR, f'X_tr_{tag}.npy')
    xte_p = os.path.join(SEC6_DIR, f'X_te_{tag}.npy')
    ytr_p = os.path.join(SEC6_DIR, f'y_tr_{tag}.npy')
    yte_p = os.path.join(SEC6_DIR, f'y_te_{tag}.npy')
    sc_p  = os.path.join(SEC6_DIR, f'scaler_{tag}.joblib')
    fi_p  = os.path.join(SEC6_DIR, f'feat_idx_{tag}.npy')

    if all(os.path.exists(p) for p in [xtr_p,xte_p,ytr_p,yte_p,sc_p,fi_p]):
        X_tr = np.load(xtr_p); X_te = np.load(xte_p)
        y_tr = np.load(ytr_p); y_te = np.load(yte_p)
        feat_idx = np.load(fi_p)
        n_cls = len(np.unique(y_tr))
        DATA[ds_name] = (X_tr, X_te, y_tr, y_te, n_cls, feat_idx)
        print(f'✓ {ds_name}: loaded '
              f'(tr={len(X_tr):,} te={len(X_te):,} cls={n_cls} feat={X_tr.shape[1]})')
        continue

    if ds_name not in RAW_DATA:
        print(f'⚠ {ds_name}: run CELL 1 first'); continue

    print(f'\nProcessing {ds_name}...')
    X_raw, y_raw = RAW_DATA[ds_name]

    # 1. Remove rare classes (< 2 samples total)
    unique, counts = np.unique(y_raw, return_counts=True)
    rare = unique[counts < 2]
    if len(rare):
        mask = ~np.isin(y_raw, rare)
        X_raw, y_raw = X_raw[mask], y_raw[mask]
        print(f'  Removed {len(rare)} rare class(es)')

    # 2. LabelEncoder → sequential [0..n-1]
    le = LabelEncoder(); y_enc = le.fit_transform(y_raw)
    joblib.dump(le, os.path.join(SEC6_DIR, f'le_{tag}.joblib'))

    # 3. SPLIT FIRST — stratified 70/30
    try:
        X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
            X_raw, y_enc, test_size=0.30, stratify=y_enc, random_state=SEED)
    except ValueError:
        X_tr_raw, X_te_raw, y_tr, y_te = train_test_split(
            X_raw, y_enc, test_size=0.30, random_state=SEED)
    print(f'  Split: train={len(X_tr_raw):,}  test={len(X_te_raw):,}')

    # 4. VarianceThreshold — fit on TRAIN only
    vt = VarianceThreshold(threshold=0.0)
    vt.fit(X_tr_raw)
    X_tr_vt = vt.transform(X_tr_raw)
    X_te_vt = vt.transform(X_te_raw)
    print(f'  After variance filter: {X_tr_vt.shape[1]} features')

    # 5. MinMaxScaler — fit on TRAIN only, transform both
    scaler = MinMaxScaler()
    X_tr_sc = scaler.fit_transform(X_tr_vt)
    X_te_sc = scaler.transform(X_te_vt)
    joblib.dump(scaler, sc_p)

    # 6. Correlation filter — computed on TRAIN only
    corrs = np.array([
        abs(np.corrcoef(X_tr_sc[:, j], y_tr)[0, 1])
        for j in range(X_tr_sc.shape[1])
    ])
    corrs = np.nan_to_num(corrs)
    feat_idx = np.where(corrs >= 0.01)[0]
    if len(feat_idx) < 5:
        feat_idx = np.argsort(corrs)[-10:]
    X_tr_fs = X_tr_sc[:, feat_idx]
    X_te_fs = X_te_sc[:, feat_idx]
    np.save(fi_p, feat_idx)
    print(f'  After correlation filter: {len(feat_idx)} features')

    # 7. SMOTE — on TRAIN only, test set untouched
    # Обмежуємо 10x oversampling щоб не роздувати до десятків мільйонів
    min_count = int(np.bincount(y_tr).min())
    max_count = int(np.bincount(y_tr).max())

    if min_count < 2:
        X_tr_sm, y_tr_sm = X_tr_fs, y_tr
        print(f'  SMOTE skipped (min class count={min_count})')
    else:
        k = min(5, min_count - 1)
        target = min(max_count, min_count * 10)
        strategy = {
            cls: target
            for cls, cnt in enumerate(np.bincount(y_tr))
            if cnt < target
        }
        if not strategy:
            X_tr_sm, y_tr_sm = X_tr_fs, y_tr
            print(f'  SMOTE skipped (classes already balanced)')
        else:
            print(f'  SMOTE: {len(strategy)} classes to oversample '
                  f'(target={target:,}, k={k})...')
            sm = SMOTE(random_state=SEED, k_neighbors=k,
                       sampling_strategy=strategy)
            try:
                X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_fs, y_tr)
                print(f'  SMOTE: {len(X_tr_fs):,} → {len(X_tr_sm):,}')
            except Exception as e:
                X_tr_sm, y_tr_sm = X_tr_fs, y_tr
                print(f'  SMOTE failed: {e}')

    n_cls = len(np.unique(y_tr_sm))

    # 8. Fix n_cls (ensure correct value)
    n_cls = len(np.unique(y_tr_sm))

    # 9. Save
    np.save(xtr_p, X_tr_sm.astype(np.float32))
    np.save(xte_p, X_te_fs.astype(np.float32))
    np.save(ytr_p, y_tr_sm)
    np.save(yte_p, y_te)
    DATA[ds_name] = (X_tr_sm.astype(np.float32),
                     X_te_fs.astype(np.float32),
                     y_tr_sm, y_te, n_cls, feat_idx)
    print(f'✓ {ds_name}: saved '
          f'(tr={len(X_tr_sm):,} te={len(X_te_fs):,} cls={n_cls} feat={len(feat_idx)})')

# Fix n_cls у DATA (критично для CNN та XGB)
for ds_name in list(DATA.keys()):
    X_tr, X_te, y_tr, y_te, n_cls_old, feat_idx = DATA[ds_name]
    n_cls_real = len(np.unique(y_tr))
    if n_cls_real != n_cls_old:
        DATA[ds_name] = (X_tr, X_te, y_tr, y_te, n_cls_real, feat_idx)
        print(f'Fixed n_cls {ds_name}: {n_cls_old} → {n_cls_real}')

print(f'\n✓ DATA ready: {len(DATA)}/4 datasets')

## CELL 3A — Train Random Forest

In [ ]:
rf_models = {}
for ds_name, (X_tr, X_te, y_tr, y_te, n_cls, _) in DATA.items():
    tag  = DS_TAGS[ds_name]
    path = os.path.join(MODELS_DIR, f'rf_{tag}.joblib')
    if os.path.exists(path):
        rf_models[ds_name] = joblib.load(path)
        acc = accuracy_score(y_te, rf_models[ds_name].predict(X_te))
        print(f'✓ RF/{ds_name}: loaded  acc={acc:.4f}')
    else:
        print(f'Training RF on {ds_name} (tr={len(X_tr):,})...')
        rf = RandomForestClassifier(
            n_estimators=100, max_features='sqrt',
            class_weight='balanced', random_state=SEED, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        joblib.dump(rf, path)
        rf_models[ds_name] = rf
        acc = accuracy_score(y_te, rf.predict(X_te))
        print(f'✓ RF/{ds_name}: acc={acc:.4f}  saved')

## CELL 3B — Train XGBoost
Для Gotham IoT 2025 використовує 500 дерев + GPU.

In [ ]:
xgb_models = {}
# Параметри по датасетах
XGB_PARAMS = {
    'CIC-IDS 2017'       : dict(n_estimators=100, max_depth=6,
                                learning_rate=0.1, tree_method='hist'),
    'UNSW-NB15'          : dict(n_estimators=100, max_depth=6,
                                learning_rate=0.1, tree_method='hist'),
    'Gotham IoT 2025'    : dict(n_estimators=500, max_depth=8,
                                learning_rate=0.05, tree_method='hist',
                                device='cuda'),   # GPU для великого датасету
    'CIC-YNU-IoTMal 2026': dict(n_estimators=100, max_depth=6,
                                learning_rate=0.1, tree_method='hist'),
}

for ds_name, (X_tr, X_te, y_tr, y_te, n_cls, _) in DATA.items():
    tag  = DS_TAGS[ds_name]
    path = os.path.join(MODELS_DIR, f'xgb_{tag}.joblib')
    if os.path.exists(path):
        xgb_models[ds_name] = joblib.load(path)
        acc = accuracy_score(y_te, xgb_models[ds_name].predict(X_te))
        print(f'✓ XGB/{ds_name}: loaded  acc={acc:.4f}')
    else:
        params = XGB_PARAMS[ds_name]
        obj = 'multi:softprob' if n_cls > 2 else 'binary:logistic'
        print(f'Training XGB on {ds_name} (tr={len(X_tr):,}, '
              f'trees={params["n_estimators"]})...')
        xgb = XGBClassifier(
            **params, subsample=0.8, colsample_bytree=0.8,
            objective=obj,
            num_class=n_cls if n_cls > 2 else None,
            eval_metric='mlogloss', random_state=SEED,
            n_jobs=-1, verbosity=1)
        xgb.fit(X_tr, y_tr,
                eval_set=[(X_te, y_te)], verbose=50)
        joblib.dump(xgb, path)
        xgb_models[ds_name] = xgb
        acc = accuracy_score(y_te, xgb.predict(X_te))
        print(f'✓ XGB/{ds_name}: acc={acc:.4f}  saved')

## CELL 3C — Train MLP (sklearn)

In [ ]:
mlp_models = {}
for ds_name, (X_tr, X_te, y_tr, y_te, n_cls, _) in DATA.items():
    tag  = DS_TAGS[ds_name]
    path = os.path.join(MODELS_DIR, f'mlp_{tag}.joblib')
    if os.path.exists(path):
        mlp_models[ds_name] = joblib.load(path)
        acc = accuracy_score(y_te, mlp_models[ds_name].predict(X_te))
        print(f'✓ MLP/{ds_name}: loaded  acc={acc:.4f}')
    else:
        print(f'Training MLP on {ds_name} (tr={len(X_tr):,})...')
        mlp = MLPClassifier(
            hidden_layer_sizes=(256,128,64), activation='relu',
            solver='adam', learning_rate_init=0.001,
            batch_size=256, max_iter=200,
            early_stopping=True, n_iter_no_change=10,
            validation_fraction=0.1, random_state=SEED)
        mlp.fit(X_tr, y_tr)
        joblib.dump(mlp, path)
        mlp_models[ds_name] = mlp
        acc = accuracy_score(y_te, mlp.predict(X_te))
        print(f'✓ MLP/{ds_name}: acc={acc:.4f}  saved')

## CELL 3D — Train CNN (PyTorch 1D)
Inference завжди по батчах (4096) — уникаємо OOM на великих датасетах.

In [ ]:
def train_cnn(X_tr, y_tr, n_feat, n_cls,
              epochs=30, batch=512, lr=0.001, patience=5):
    model = CNN1D(n_feat, n_cls).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    crit  = nn.CrossEntropyLoss()
    dl    = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
        batch_size=batch, shuffle=True, pin_memory=False)
    best, no_imp = float('inf'), 0
    for epoch in range(epochs):
        model.train(); total = 0
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward(); opt.step()
            total += loss.item()
        avg = total / len(dl)
        if avg < best - 1e-4: best = avg; no_imp = 0
        else: no_imp += 1
        if no_imp >= patience:
            print(f'  Early stop epoch {epoch+1}'); break
    model.eval(); return model

cnn_models = {}
for ds_name, (X_tr, X_te, y_tr, y_te, n_cls, _) in DATA.items():
    tag      = DS_TAGS[ds_name]
    pt_path  = os.path.join(MODELS_DIR, f'cnn_{tag}.pt')
    cfg_path = os.path.join(MODELS_DIR, f'cnn_{tag}_cfg.json')

    # Очищаємо GPU між датасетами
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
        import gc; gc.collect()

    if os.path.exists(pt_path) and os.path.exists(cfg_path):
        with open(cfg_path) as f: c = json.load(f)
        m = CNN1D(c['n_features'], c['n_classes']).to(DEVICE)
        m.load_state_dict(torch.load(pt_path, map_location=DEVICE))
        m.eval(); cnn_models[ds_name] = m
        preds = predict_batched(m, X_te)
        acc   = accuracy_score(y_te, preds)
        print(f'✓ CNN/{ds_name}: loaded  acc={acc:.4f}')
    else:
        print(f'Training CNN on {ds_name} (tr={len(X_tr):,})...')
        n_feat = X_tr.shape[1]
        m = train_cnn(X_tr, y_tr, n_feat, n_cls)
        torch.save(m.state_dict(), pt_path)
        with open(cfg_path, 'w') as f:
            json.dump({'n_features': n_feat, 'n_classes': n_cls}, f)
        cnn_models[ds_name] = m
        preds = predict_batched(m, X_te)
        acc   = accuracy_score(y_te, preds)
        print(f'✓ CNN/{ds_name}: trained  acc={acc:.4f}  saved')

## CELL 4 — Evaluate: Table 6 v2

In [ ]:
CSV_PATH = os.path.join(SEC6_DIR, 'table6_v2.csv')

if os.path.exists(CSV_PATH):
    df_results = pd.read_csv(CSV_PATH)
    print('✓ Table 6 v2 loaded from Drive:')
    print(df_results.to_string(index=False))
else:
    MODEL_GROUPS = {
        'RF': rf_models, 'XGBoost': xgb_models,
        'MLP': mlp_models, 'CNN': cnn_models,
    }

    def get_proba(model, X, mtype, batch=4096):
        """Inference по батчах для CNN щоб уникнути OOM."""
        if mtype == 'CNN':
            model.eval()
            all_proba = []
            with torch.no_grad():
                for i in range(0, len(X), batch):
                    xb = torch.FloatTensor(X[i:i+batch]).to(DEVICE)
                    pb = torch.softmax(model(xb), 1).cpu().numpy()
                    all_proba.append(pb)
                    del xb, pb
            if DEVICE == 'cuda': torch.cuda.empty_cache()
            return np.concatenate(all_proba)
        return model.predict_proba(X)

    def safe_auc(y_te, proba):
        unique, counts = np.unique(y_te, return_counts=True)
        valid = unique[counts >= 2]
        aucs = []
        for c in valid:
            yb = (y_te == c).astype(int)
            if len(np.unique(yb)) < 2: continue
            try: aucs.append(roc_auc_score(yb, proba[:, c]))
            except: pass
        return float(np.mean(aucs)) if aucs else float('nan')

    rows = []
    for ds_name, (X_tr, X_te, y_tr, y_te, n_cls, _) in DATA.items():
        row = {'Dataset': ds_name}
        for mname, models in MODEL_GROUPS.items():
            if ds_name not in models: continue
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
                import gc; gc.collect()
            proba  = get_proba(models[ds_name], X_te, mname)
            y_pred = np.argmax(proba, axis=1)
            acc = accuracy_score(y_te, y_pred)
            f1  = f1_score(y_te, y_pred, average='macro', zero_division=0)
            auc = safe_auc(y_te, proba)
            row[f'{mname}_Acc'] = round(acc, 4)
            row[f'{mname}_F1']  = round(f1,  4)
            row[f'{mname}_AUC'] = round(auc, 4)
            print(f'{ds_name:25s} | {mname:8s} | '
                  f'Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
            del proba
        rows.append(row)

    df_results = pd.DataFrame(rows)
    df_results.to_csv(CSV_PATH, index=False)
    print(f'\n✓ Table 6 v2 saved → {CSV_PATH}')

## CELL 5 — Figure 5 v2: ROC curves

In [ ]:
OUT5 = os.path.join(FIG_DIR, 'fig5_v2_roc_curves.png')

if os.path.exists(OUT5):
    print('✓ Figure 5 v2 exists — delete to regenerate')
else:
    import gc
    MODEL_GROUPS = {
        'RF': rf_models, 'XGBoost': xgb_models,
        'MLP': mlp_models, 'CNN': cnn_models,
    }
    COLORS = {'RF':'#2878B5','XGBoost':'#C82423',
               'MLP':'#FF8C00','CNN':'#6A5ACD'}
    PANELS = ['(a)','(b)','(c)','(d)']

    def get_proba_fig(model, X, mtype, batch=4096):
        if mtype == 'CNN':
            model.eval(); all_p = []
            with torch.no_grad():
                for i in range(0, len(X), batch):
                    xb = torch.FloatTensor(X[i:i+batch]).to(DEVICE)
                    pb = torch.softmax(model(xb),1).cpu().numpy()
                    all_p.append(pb); del xb, pb
            if DEVICE=='cuda': torch.cuda.empty_cache()
            return np.concatenate(all_p)
        return model.predict_proba(X)

    fig, axes = plt.subplots(2,2, figsize=(FIG_W, FIG_W*0.85))
    fig.subplots_adjust(hspace=0.42, wspace=0.38)

    for idx,(ds_name,(X_tr,X_te,y_tr,y_te,n_cls,_)) in enumerate(DATA.items()):
        ax = axes[idx//2][idx%2]
        u,c = np.unique(y_te, return_counts=True)
        valid_cls = u[c>=2]

        for mname, models in MODEL_GROUPS.items():
            if ds_name not in models: continue
            if DEVICE=='cuda': torch.cuda.empty_cache(); gc.collect()
            proba = get_proba_fig(models[ds_name], X_te, mname)
            try:
                all_fpr = np.linspace(0,1,300)
                mean_tpr = np.zeros(300); n_v=0
                for c in valid_cls:
                    yb=(y_te==c).astype(int)
                    if len(np.unique(yb))<2: continue
                    fpr_c,tpr_c,_=roc_curve(yb, proba[:,c])
                    mean_tpr+=np.interp(all_fpr,fpr_c,tpr_c); n_v+=1
                if n_v: mean_tpr/=n_v
                aucs=[roc_auc_score((y_te==c).astype(int),proba[:,c])
                      for c in valid_cls
                      if len(np.unique((y_te==c).astype(int)))==2]
                auc_v=float(np.mean(aucs)) if aucs else float('nan')
                ax.plot(all_fpr, mean_tpr, color=COLORS[mname],
                        lw=0.9, label=f'{mname} (AUC={auc_v:.3f})')
            except Exception as e:
                print(f'  ROC error {ds_name}/{mname}: {e}')
            del proba

        ax.plot([0,1],[0,1],color='#AAAAAA',lw=0.6,ls='--')
        ax.set_xlabel('False Positive Rate',fontsize=7,labelpad=2)
        ax.set_ylabel('True Positive Rate',fontsize=7,labelpad=2)
        ax.set_title(f'{PANELS[idx]} {ds_name}',fontsize=8,pad=4,loc='left')
        ax.legend(fontsize=5.5,loc='lower right',handlelength=1.2)
        ax.set_xlim(0,1); ax.set_ylim(0,1.01)
        ax.tick_params(labelsize=6,length=2.5,width=0.5)
        for sp in ['top','right']: ax.spines[sp].set_visible(False)

    plt.savefig(OUT5, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'✓ Figure 5 v2 saved → {OUT5}')

## CELL 6 — Збереження та верифікація масивів для Section 7

In [ ]:
print('Verifying saved arrays for Section 7...')
for ds_name, (X_tr, X_te, y_tr, y_te, n_cls, feat_idx) in DATA.items():
    tag = DS_TAGS[ds_name]
    for fname, arr in [
        (f'X_tr_{tag}.npy', X_tr), (f'X_te_{tag}.npy', X_te),
        (f'y_tr_{tag}.npy', y_tr), (f'y_te_{tag}.npy', y_te),
        (f'feat_idx_{tag}.npy', feat_idx),
    ]:
        p = os.path.join(SEC6_DIR, fname)
        if not os.path.exists(p):
            np.save(p, arr); print(f'  saved: {fname}')
        else:
            kb = os.path.getsize(p)/1024
            print(f'  ✓ {fname:<50} {kb:>8.0f} KB')
print('\n✓ All arrays ready for Section7_Adversarial_v2.ipynb')

## CELL 7 — Final Status

In [ ]:
print('=== SECTION 6 v2 FINAL STATUS ===')
all_ok = True
checks = {
    'Table 6 v2'  : os.path.join(SEC6_DIR, 'table6_v2.csv'),
    'Figure 5 v2' : os.path.join(FIG_DIR,  'fig5_v2_roc_curves.png'),
}
for tag in DS_TAGS.values():
    for prefix in ['X_tr','X_te','y_tr','y_te','feat_idx']:
        checks[f'{prefix}_{tag}'] = os.path.join(SEC6_DIR, f'{prefix}_{tag}.npy')
    checks[f'scaler_{tag}'] = os.path.join(SEC6_DIR, f'scaler_{tag}.joblib')
    for mn in ['rf','xgb','mlp']:
        checks[f'{mn}_{tag}'] = os.path.join(MODELS_DIR, f'{mn}_{tag}.joblib')
    checks[f'cnn_{tag}'] = os.path.join(MODELS_DIR, f'cnn_{tag}.pt')

for label, path in checks.items():
    ok = os.path.exists(path)
    kb = os.path.getsize(path)/1024 if ok else 0
    icon = '✓' if ok else '✗ MISSING'
    print(f'  {icon:<12} {label:<48} {kb:>8.0f} KB')
    if not ok: all_ok = False

print()
if all_ok:
    print('✓ All outputs ready. Run Section7_Adversarial_v2.ipynb next.')
else:
    print('⚠ Missing files — rerun the corresponding CELL above.')